<a href="https://colab.research.google.com/github/ramhalghamdy-source/AI-Powered-Phage-Therapy-Recommendation-System/blob/main/AI_Powered_Phage_Therapy_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install biopython xgboost==1.7.6 shap==0.43.0 numpy pandas scikit-learn

from Bio import Entrez, SeqIO
import pandas as pd, numpy as np, io, itertools, math, json, os, textwrap
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from xgboost import XGBClassifier
from collections import Counter
from tqdm.auto import tqdm


Entrez.email = "ramh.alghamdy@gmail.com"
ENTREZ_API_KEY = "756f21b89f56790808cdbc0d764c6e184608"
if ENTREZ_API_KEY:
    Entrez.api_key = ENTREZ_API_KEY


def try_esearch(db, terms, retmax=100):

    for term in terms:
        try:
            with Entrez.esearch(db=db, term=term, retmax=retmax) as h:
                r = Entrez.read(h)
            ids = r.get("IdList", [])
            print(f"[OK] {db} ids={len(ids)} | term: {term}")
            if ids:
                return ids, term
        except Exception as e:
            print(f"[ERR] term failed: {term} -> {e}")
    return [], None

def fetch_fasta_dict(db, ids):
    if not ids: return {}

    out = {}
    step = 200
    for i in range(0, len(ids), step):
        chunk = ids[i:i+step]
        with Entrez.efetch(db=db, id=",".join(chunk), rettype="fasta", retmode="text") as h:
            data = h.read()
        recs = list(SeqIO.parse(io.StringIO(data), "fasta"))
        out.update({r.id: r for r in recs})
    return out


bac_terms = [
    'txid1280[Organism:exp] AND (complete genome[All Fields] OR chromosome[Title] OR complete sequence[All Fields])',
    '"Staphylococcus aureus"[Organism] AND (complete genome[All Fields] OR chromosome[Title])',
    '"Staphylococcus aureus"[All Fields] AND genome[Title]',
]
bac_ids, used_bac = try_esearch("nuccore", bac_terms, retmax=30)
bacteria = fetch_fasta_dict("nuccore", bac_ids)
print("bacteria:", len(bacteria))


phage_pos_terms = [
    '"Staphylococcus phage"[Organism] AND (complete genome[All Fields] OR complete sequence[All Fields])',
    '(phage[Title] OR bacteriophage[Title]) AND Staphylococcus[All Fields] AND (complete genome[All Fields] OR complete sequence[All Fields])',
    '("Staphylococcus"[All Fields] AND phage[All Fields]) AND genome',
]
ph_pos_ids, used_pos = try_esearch("nuccore", phage_pos_terms, retmax=200)
phages_pos = fetch_fasta_dict("nuccore", ph_pos_ids)
print("phages_pos:", len(phages_pos))


phage_neg_terms = [
    '"Escherichia phage"[Organism] AND (complete genome[All Fields] OR complete sequence[All Fields])',
    '"Pseudomonas phage"[Organism] AND (complete genome[All Fields] OR complete sequence[All Fields])',
    '(phage[Title]) AND (Escherichia[All Fields] OR Pseudomonas[All Fields]) AND genome',
]
ph_neg_ids, used_neg = try_esearch("nuccore", phage_neg_terms, retmax=200)
phages_neg = fetch_fasta_dict("nuccore", ph_neg_ids)
print("phages_neg:", len(phages_neg))


print("\n=== SUMMARY ===")
print("Bacteria   :", len(bacteria), "| query used:", used_bac)
print("Phage (+)  :", len(phages_pos), "| query used:", used_pos)
print("Phage (-)  :", len(phages_neg), "| query used:", used_neg)

import numpy as np
from math import sqrt
from numpy.linalg import norm
from tqdm.auto import tqdm



MAX_BAC   = 12
MAX_PH_POS= 60
MAX_PH_NEG= 60

def take_subset(d, max_n):
    keys = list(d.keys())[:max_n]
    return {k: d[k] for k in keys}

bacteria_sub   = take_subset(bacteria,   MAX_BAC)
phages_pos_sub = take_subset(phages_pos, MAX_PH_POS)
phages_neg_sub = take_subset(phages_neg, MAX_PH_NEG)

print(len(bacteria_sub), len(phages_pos_sub), len(phages_neg_sub))


ALPH = ["A","C","G","T"]
def all_kmers(k=6):
    kmers = ['']
    for _ in range(k):
        kmers = [p+c for p in kmers for c in ALPH]
    return kmers

K = 6
KMERS = all_kmers(K)
IDX = {k:i for i,k in enumerate(KMERS)}

def seq_to_kmer_vec(seq):
    s = str(seq).upper()
    v = np.zeros(len(KMERS), dtype=np.float32)

    for i in range(0, len(s)-K+1):
        k = s[i:i+K]
        if set(k) <= set("ACGT"):
            v[IDX[k]] += 1
    ssum = v.sum()
    if ssum > 0:
        v /= ssum
    return v

def gc_content(seq):
    s = str(seq).upper()
    g = s.count("G"); c = s.count("C")
    a = s.count("A"); t = s.count("T")
    tot = a+t+g+c
    return (g+c)/tot if tot else 0.0

def compute_features(records):
    feats = {}
    for rid, rec in tqdm(records.items(), desc="extract_features"):
        feats[rid] = {
            "kmer": seq_to_kmer_vec(rec.seq),
            "gc":   gc_content(rec.seq)
        }
    return feats

bac_feats   = compute_features(bacteria_sub)
ph_pos_feats= compute_features(phages_pos_sub)
ph_neg_feats= compute_features(phages_neg_sub)

len(bac_feats), len(ph_pos_feats), len(ph_neg_feats)

import pandas as pd

def cosine(a,b):
    na, nb = norm(a), norm(b)
    if na==0 or nb==0: return 0.0
    return float(np.dot(a,b)/(na*nb))

rows = []


for ph_id, ph in ph_pos_feats.items():
    for bac_id, bc in bac_feats.items():
        rows.append({
            "phage_id": ph_id,
            "bacteria_id": bac_id,
            "cosine_kmer_sim": cosine(ph["kmer"], bc["kmer"]),
            "gc_diff": abs(ph["gc"] - bc["gc"]),
            "label": 1
        })


for ph_id, ph in ph_neg_feats.items():
    for bac_id, bc in bac_feats.items():
        rows.append({
            "phage_id": ph_id,
            "bacteria_id": bac_id,
            "cosine_kmer_sim": cosine(ph["kmer"], bc["kmer"]),
            "gc_diff": abs(ph["gc"] - bc["gc"]),
            "label": 0
        })

df = pd.DataFrame(rows)
df.sample(5), df["label"].value_counts()

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from xgboost import XGBClassifier

X = df[["cosine_kmer_sim","gc_diff"]].values
y = df["label"].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = XGBClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
    random_state=42
)
model.fit(X_tr, y_tr)

probs = model.predict_proba(X_te)[:,1]
preds = (probs >= 0.5).astype(int)

print("AUC-ROC:", round(roc_auc_score(y_te, probs), 3))
print("F1-score:", round(f1_score(y_te, preds), 3))
print(classification_report(y_te, preds, digits=3))

phage_ids  = list(ph_pos_feats.keys()) + list(ph_neg_feats.keys())
strain_ids = list(bac_feats.keys())


def prob_matrix_for_strains(strain_list):
    P_rows = []
    for sid in strain_list:
        bc = bac_feats[sid]
        feats_mat = []
        for pid in phage_ids:
            ph = ph_pos_feats.get(pid, ph_neg_feats.get(pid))
            feats_mat.append([cosine(ph["kmer"], bc["kmer"]),
                              abs(ph["gc"] - bc["gc"])])
        feats_mat = np.array(feats_mat)
        probs = model.predict_proba(feats_mat)[:,1]
        P_rows.append(probs)
    P = np.vstack(P_rows).T
    return P

def greedy_cocktail(P, prob_thresh=0.7, max_phages=3):
    covered = np.zeros(P.shape[1], dtype=bool)
    chosen = []
    for _ in range(max_phages):
        gains = [np.sum((P[i] >= prob_thresh) & (~covered)) for i in range(P.shape[0])]
        i = int(np.argmax(gains))
        if gains[i] == 0:
            break
        chosen.append(i)
        covered |= (P[i] >= prob_thresh)
    return [phage_ids[i] for i in chosen], covered


example_strains = strain_ids[:3]
P = prob_matrix_for_strains(example_strains)
cocktail, covered = greedy_cocktail(P, prob_thresh=0.7, max_phages=3)

print("Strains used:", [s.split('.')[0] for s in example_strains])
print("Suggested cocktail (≤3 phages):")
for pid in cocktail:
    print(" -", pid)
print("Coverage by threshold 0.7:", covered.tolist())

def report_for_strains(chosen_strains, top_k=5):
    P = prob_matrix_for_strains(chosen_strains)
    out = []
    for s_idx, sid in enumerate(chosen_strains):
        table = pd.DataFrame({"phage_id": phage_ids, "prob": P[:, s_idx]})
        out.append((sid, table.sort_values("prob", ascending=False).head(top_k)))
    cocktail, covered = greedy_cocktail(P, 0.7, 3)
    return out, cocktail

report, cocktail = report_for_strains(strain_ids[:2], top_k=5)
for sid, tbl in report:
    print("\n=== Top phages for strain:", sid, "===")
    display(tbl.reset_index(drop=True))
print("\nSuggested cocktail:", cocktail)